In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import T5ForConditionalGeneration, AutoTokenizer, Trainer, TrainingArguments
import transformers
from sklearn.model_selection import train_test_split
import time
import matplotlib.pyplot as plt
import evaluate
import random
import math
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [2]:
data = pd.read_csv('personality_gold_summary.csv', index_col = 0)#.sample(frac = 1).reset_index(drop = True)
data.head()

,persona,chat,summary
conversation number,,,
1,i like to remodel homes. i like to go hunting...,"hi , how are you doing ? i am getting ready to...",The user enjoys remodeling homes and bow hunti...
2,my mom is my best friend. i have four sisters...,"hi , how are you doing today ?\ni am spending ...","The user, who has a close relationship with th..."
3,i had a gig at local theater last night. i wo...,"we all live in a yellow submarine , a yellow s...",The conversation features a stand-up comedian ...
4,i am very athletic. i wear contacts. i have b...,hi ! i work as a gourmet cook .\ni do not like...,The conversation features an athletic individu...
5,i am primarily a meat eater. i am a guitar pl...,how are you doing today\nwhat do you do for ca...,"The user, a meat eater and guitar player who w..."


In [3]:
size = len(data)
print("Size of the dataset = ", size)


Size of the dataset =  2000


In [4]:
prompting_technique = 'zero-shot'     #'cot', 'few-shot' or 'zero-shot' are the three keywords for using the three techniques

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
import torch
import accelerate
# Custom Dataset Class
class PersonaChatDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.tokenizer = tokenizer
        self.text = {'input': [], 'label': []}

        for i in list(dataframe.index):
            persona = str(dataframe['persona'][i])
            chat = str(dataframe['chat'][i])
            summary = str(dataframe['summary'][i])

            prompt = (
                "Summarize the following conversation history concisely while preserving and keeping in mind key details such as persona of the user. "
                "Output only the summary in a paragraph.\n"
                f"Persona: {persona}\n"
                f"Conversation History: {chat}\n\nYour answer:"
            )

            if not prompt or not summary:
                continue

            self.text['input'].append(prompt)
            self.text['label'].append(summary)

        self.m = len(self.text['input'])

        self.input_ids = []
        for input_text, label_text in zip(self.text['input'], self.text['label']):
            tokenized_text = tokenizer(
                input_text + " " + label_text,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=1024  # adjust as needed
            )
            self.input_ids.append(tokenized_text['input_ids'].squeeze())

    def __getitem__(self, idx):
        return {'input_ids': self.input_ids[idx]}

    def __len__(self):
        return self.m

# Replace with your actual Hugging Face token
access_token = "REDACTED"

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained('google/gemma-3-1b-it', use_auth_token=access_token)
tokenizer = AutoTokenizer.from_pretrained('google/gemma-3-1b-it', use_auth_token=access_token)

# Split dataset
train_df, val_df = train_test_split(data, test_size=0.2, random_state=42)

# train_df, teste_df = train_test_split(data, test_size=0.2, random_state=42)
# test_df, val_df= train_test_split(teste_df, test_size=0.5, random_state=42)
# Create datasets
train_dataset = PersonaChatDataset(train_df, tokenizer)
val_dataset = PersonaChatDataset(val_df, tokenizer)

# Training arguments
output_dir = './gemma-2b-personachat-summary'

training_args = TrainingArguments(
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    warmup_steps=2,
    num_train_epochs=3,
    learning_rate=2e-5,
    max_steps=2500,
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=100,
    output_dir=output_dir,
    optim="adamw_torch",
    overwrite_output_dir=True,
    logging_dir="./logs",
    report_to="none"  # or "tensorboard" or "wandb" if you're using those
)


# Data collator (no MLM for causal models)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Trainer
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    data_collator=data_collator
)

# Start training
trainer.train()


/opt/conda/lib/python3.11/site-packages/transformers/models/auto/auto_factory.py:476: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/transformers/models/auto/tokenization_auto.py:897: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
It is strongly recommended to train Gemma3 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.


Step,Training Loss,Validation Loss
100,2.357900,2.242751
200,2.197900,2.202068
300,2.221300,2.146208
400,2.129800,2.121385
500,2.113300,2.082051
600,2.073400,2.061827
700,2.038100,2.045217
800,2.025900,2.018489
900,1.998400,1.997621
1000,2.026200,1.983826


TrainOutput(global_step=2500, training_loss=1.764222750854492, metrics={'train_runtime': 1913.3684, 'train_samples_per_second': 1.307, 'train_steps_per_second': 1.307, 'total_flos': 3793201895468544.0, 'train_loss': 1.764222750854492, 'epoch': 1.5625})

In [8]:
from tqdm import tqdm
import torch
import pandas as pd

# Make sure you're on the right device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

predicted_answers = []
real_answers = []

for idx in tqdm(range(len(val_dataset)), desc="Processing validation data"):
    with torch.no_grad():
        torch.cuda.empty_cache()

        # Extract prompt without gold summary
        persona = str(val_df.iloc[idx]['persona'])
        chat = str(val_df.iloc[idx]['chat'])
        gold_summary = str(val_df.iloc[idx]['summary'])

        prompt = (
            "Summarize the following conversation history concisely while preserving and keeping in mind key details such as persona of the user. "
            "Output only the summary in a paragraph.\n"
            f"Persona: {persona}\n"
            f"Conversation History: {chat}\n\nYour answer:"
        )

        # Tokenize only the prompt
        input_ids = tokenizer(prompt, return_tensors='pt', truncation=True, padding=True).input_ids.to(device)

        # Generate prediction
        output = model.generate(
            input_ids=input_ids,
            max_new_tokens=150,
            do_sample=False  # deterministic output (optional)
        )

        decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)

        # Optional cleanup if the prompt is echoed in the output
        if decoded_output.startswith(prompt):
            decoded_output = decoded_output[len(prompt):].strip()

        # Save outputs
        predicted_answers.append(decoded_output)
        real_answers.append(gold_summary)

        print(f"\nActual Summary: {gold_summary}")
        print(f"Predicted Summary: {decoded_output}\n")

# Save to CSV
df = pd.DataFrame({
    'Actual': real_answers,
    'Predicted': predicted_answers
})
df.to_csv('predicted_vs_actual_gemma_2500.csv', index=False)
print("CSV file 'predicted_vs_actual_gemma_2500.csv' created successfully.")


Processing validation data:   0%|          | 1/400 [00:07<50:35,  7.61s/it]


Actual Summary: The conversation features a preschool girl who enjoys playing outside, dancing, and loves zebras. She shares her excitement about her day at preschool and her recent birthday party at Chuck E. Cheese. In contrast, an adult expresses stress from a demanding job as a restaurant manager, mentioning long hours and difficult customers. The girl expresses hope to continue dancing or working with zebras when she grows up, while the adult questions her online presence, to which she responds that her mom allows her 20 minutes of screen time daily.
Predicted Summary: A preschool girl shares her excitement about her day in preschool, expressing her love for playing outside and dancing, even though she works 60 hours a week as a restaurant manager. She mentions her stressful job and her boss's poor treatment, while also highlighting her aspirations to dance professionally and work with zebras. The conversation touches on her hopes for a higher salary and her limited screen time, a

Processing validation data:   0%|          | 2/400 [00:15<50:18,  7.58s/it]


Actual Summary: The user, who enjoys Isaiah Rashad's music and prefers reading over watching movies, engages in a friendly conversation about music, expressing a preference for upbeat genres and rap. They mention a specific song featuring Kendrick Lamar and discuss their busy schedule due to their marketing job, which limits their exercise routine.
Predicted Summary: The user, who enjoys music by Isaiah Rashad and prefers reading books over watching movies, engages in a light conversation about music preferences. They express interest in rap music, mentioning artists like Kendrick Lamar, and discuss their busy marketing job that limits their time for exercise. The conversation also touches on the user's appreciation for running music, which they plan to explore further. They clarify that while they enjoy music, their job keeps them busy and prevents them from exercising. The other person shares their own interests in music and suggests a potential connection between them and the user.

Processing validation data:   1%|          | 3/400 [00:22<50:10,  7.58s/it]

...


Actual Summary: The conversation features a user who recently graduated with a nursing degree and enjoys staying in shape, discussing fitness with another individual who is 35 years old and recently experienced an unexpected divorce. The user expresses sympathy for the other person's situation while sharing their own age of 33 and mentioning their love for exercise.
Predicted Summary: The user, a 35-year-old nursing graduate with a husband who is a firefighter, shares that they are doing okay and recently went for a run. They express a love for staying in shape and mention having more time since their wife left them at 33. The conversation partner, a 33-year-old, reveals they are 35 and recently turned 33 in August. The user empathizes with the partner's situation and inquires about their ages. The partner mentions they are not looking for anything specific at the moment but are open to meeting new people. The user responds positively, showing interest in getting to know the partner b

Processing validation data: 100%|█████████▉| 399/400 [50:44<00:07,  7.66s/it]


Actual Summary: The user, a middle child and babysitter with a sweet tooth, shares that they are enjoying candy while reminiscing about their grandfather, who gifted each of his three granddaughters a car, including the user's Mercedes. They mention their past achievements in running track, though they no longer compete, and express their love for making treats for the kids they babysit, particularly chocolate chip cookies, which they attribute to their grandfather's influence.
Predicted Summary: The user, a middle child babysitting and a sweets tooth, shares that they enjoy a big bag of candy and have a sweet tooth. They mention their Mercedes and their experience babysitting, noting they have a lot of trophies from running track due to their age. The conversation touches on the user's grandfather, who had three cars and gifted them one, including his sweet tooth. They express appreciation for the user's chocolate chip cookies, which their grandfather also enjoyed. The user highlight

Processing validation data: 100%|██████████| 400/400 [50:52<00:00,  7.63s/it]


Actual Summary: The user, who enjoys metal music and watching TV and movies, is currently relaxing at home with a Walking Dead marathon. They suggest Breaking Bad as a more interesting alternative and mention an upcoming comic convention. The conversation reveals that the user is from Georgia and is chatting with someone from Boston, expressing a fondness for the Boston accent and noting they have friends from there.
Predicted Summary: The user, who enjoys watching TV and movies, is having a good weekend relaxing with movies and beer. They are planning a "Walking Dead marathon" and mention their comic convention coming up. The conversation partner is from Georgia and has friends from Boston, leading to a light-hearted exchange about accents. The user expresses interest in the show "It" and shares that they have friends from Boston, while the partner mentions they have friends from Georgia. The conversation highlights their shared interests and the user's background in advertising. The